# Lesson 7: The Transformer's superpower — attention 👀
**Learning goals**
1. Understand "attention": when handling one word, look at which other words matter most
2. Calculate attention by hand with numpy
3. Open up a real model and draw its "attention map"

**⏱️ Timing:** analogy 10' | attention by hand 25' | real model 20' | discussion 5'

## 🎧 An analogy first
Read this: **"The trophy didn't fit in the suitcase because it was too big."** What is "it"?
You know straight away it's the trophy, because your attention **jumps back** to find the related words.

The rhyme writer from Lesson 5 only looked at one previous word, and the RNNs from Lesson 6 had to pass their memory along step by step, like a telephone game.
The **Transformer** (the T in GPT) has a secret: every word gives every other word a "relevance score", then pays most attention to the high scores.

## Part 1: Attention by hand 🧮
Each word is represented by a list of numbers (a **vector**) — think of it as the word's "personality traits". The more similar two vectors are, the bigger their **dot product** = the more related they are.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# We invent 3 traits for each word: [is it an object?, is it a person?, related to "big"?]
words = ["Sam", "trophy", "suitcase", "it", "big"]
vec = np.array([
    [0.1, 1.0, 0.0],   # Sam: a person
    [1.0, 0.0, 0.9],   # trophy: an object, can be big
    [1.0, 0.0, 0.4],   # suitcase: an object
    [0.9, 0.1, 0.6],   # it: refers to an object
    [0.2, 0.0, 1.0],   # big
])

# How related is "it" to each word? → dot product
scores = vec @ vec[3]
for w, s in zip(words, scores):
    print(f"it ↔ {w}: {s:.2f}")

Raw scores are hard to compare, so we use **softmax** to turn them into "attention shares" that add up to 100%:

In [ ]:
def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

attn = softmax(scores)
for w, a in zip(words, attn):
    print(f"{w:9} {'█' * int(a * 50)} {a:.0%}")

In [ ]:
# Every word looking at every word: the attention matrix
matrix = np.array([softmax(vec @ v) for v in vec])
plt.imshow(matrix, cmap="Blues")
plt.xticks(range(5), words); plt.yticks(range(5), words)
plt.colorbar(); plt.title("who pays attention to whom"); plt.show()

### 🧩 Exercise 1
1. Change Sam's vector to `[0.9, 1.0, 0.8]` (make Sam "very object-like and big"). How does the attention of "it" change? What does that tell you?
2. **Key point:** in a real model, nobody invents these numbers — the model **learns them during training**.

(Seniors: real attention first turns each vector into a Query, a Key and a Value, and the formula is softmax(QKᵀ/√d)·V. But the core idea is exactly the dot product + softmax above.)

## Part 2: Look inside a real model's brain 🧠
We'll use BERT (a Transformer model) to read a sentence and plot its attention.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

def show_attention(sentence, layer=8, word="it"):
    inputs = tok(sentence, return_tensors="pt")
    with torch.no_grad():
        att = bert(**inputs).attentions[layer][0].mean(0)   # average over all "heads" in this layer
    tokens = tok.convert_ids_to_tokens(inputs.input_ids[0])
    i = tokens.index(word)
    plt.figure(figsize=(10, 2))
    plt.bar(tokens, att[i].numpy())
    plt.xticks(rotation=45); plt.title(f"layer {layer}: where does '{word}' look?"); plt.show()

show_attention("the animal didn't cross the street because it was too tired")

### 🧩 Exercise 2: Change one word, change what "it" means
Swap `tired` for `wide`. Now "it" should mean the street. Does the model's attention follow?
(Note: the model isn't perfect — sometimes the change is small. That itself shows the AI doesn't truly "understand".)

In [ ]:
show_attention("the animal didn't cross the street because it was too wide")

### 🚀 Challenge
1. Change `layer` (0 to 11). How is attention in early layers different from deep layers?
2. Write your own sentence with a pronoun (he / she / it / they). Can the model find the right word?

In [ ]:
# challenge area

## 📌 Summary
- **Attention** = each word scores how relevant every other word is, then focuses on the high scores
- Dot product measures similarity; softmax turns scores into percentages
- GPT = **G**enerative **P**re-trained **T**ransformer (a model built around attention)
- **Next lesson:** we open up the Transformer — Query, Key, Value, positions and multiple heads